In [525]:
import sys
from PIL import Image, ImageDraw, ImageFont
import pickle as pkl
import textwrap
from matplotlib import font_manager
from tqdm import tqdm
import json
from transformers import AutoTokenizer
import os

json_path = (
    "/share/kuleshov/ma2238/dllm-dev-new/dllm-dev/output/intermediate_samples_e2d2.json"
)
intm_samples_e2d2 = json.loads(open(json_path).read())[-1]

json_path = (
    "/share/kuleshov/ma2238/dllm-dev-new/dllm-dev/output/intermediate_samples_bd3.json"
)
intm_samples_bd3lm = json.loads(open(json_path).read())[4]

tokenizer = AutoTokenizer.from_pretrained("gpt2")

In [526]:
from PIL import Image, ImageDraw, ImageFont
from matplotlib import font_manager
from tqdm import tqdm
from transformers import AutoTokenizer
import os, textwrap, math
import random

random.seed(42)  # pick any integer

# ---- Tokenizer (use your model/tokenizer of choice) ----
# If you're already passing a tokenizer from elsewhere, remove this line.
tokenizer = AutoTokenizer.from_pretrained("gpt2", use_fast=True)


def draw_image(
    prompt,
    text,
    wrapped_text,
    is_correct,
    is_enc_dec,
    block_size=4,
    img_width=1100,
    img_height=500,
    caching_step=False,
    mask_token="",
    scale=2,  # 2x or 3x recommended for hi-res
    downscale_output=False,  # supersample then downscale back to (img_width, img_height)
):
    s = max(1, int(scale))

    # ---- Fonts ----
    prompt_font = ImageFont.truetype(
        font_manager.findfont(font_manager.FontProperties(family="monospace")), 20 * s
    )
    base_path = font_manager.findfont(font_manager.FontProperties(family="monospace"))
    font = ImageFont.truetype(base_path, 25 * s)
    possible_bold_fonts = [
        "/usr/share/fonts/truetype/dejavu/DejaVuSansMono-Bold.ttf",
        "/usr/share/fonts/truetype/liberation/LiberationMono-Bold.ttf",
        "C:/Windows/Fonts/consolab.ttf",
        "C:/Windows/Fonts/courbd.ttf",
        "/Library/Fonts/Courier New Bold.ttf",
    ]
    bold_font_path = next((f for f in possible_bold_fonts if os.path.exists(f)), None)
    bold_font = ImageFont.truetype(
        bold_font_path if bold_font_path else base_path, 25 * s
    )

    # ---- Style / layout ----
    line_height = 1.8
    # Wrap prompt to canvas width (coarse estimate using monospace-ish width)
    avg_char_px = max(6, int(10 * s))
    prompt_wrap_cols = int(img_width * 0.7) // avg_char_px
    prompt_wrapped = textwrap.fill(prompt, width=prompt_wrap_cols)

    box_fill = (214, 234, 248)
    border_color = (0, 0, 0)
    border_width_thin = max(1, int(1 * s))
    border_width_thick = max(1, int((1 if is_enc_dec else 3) * s))
    corner_radius = int(5 * s)
    x_margin = int(25 * s)
    x_padding = int(1 * s)
    y_padding = int(4 * s)
    extra_bottom = int(4 * s)  # stretch downward
    dot_spacing = int(8 * s)
    dot_radius = max(1, int(1 * s))

    # ---- Canvas (use provided width/height) ----
    canvas_w = int(img_width)
    canvas_h = int(img_height)
    # If supersampling, render bigger and later downscale:
    render_w = canvas_w * (1 if not downscale_output else s)
    render_h = canvas_h * (1 if not downscale_output else s)

    img = Image.new("RGB", (int(render_w * 1.6), render_h), color=(255, 255, 255))
    d = ImageDraw.Draw(img)

    # ---- Tokens for the FULL text (for global indexing) ----
    detok_text = tokenizer.batch_decode(
        tokenizer(text, return_tensors="pt")["input_ids"][0]
    )

    # ------------ FIXED GLOBAL ELLIPSIS / CONTENT LENGTH ---------------- #
    # Count ONLY trailing ' .' so the latest-window doesn't slide as dots stream in
    trailing_dots = 0
    for t in reversed(detok_text):
        if t == " .":
            trailing_dots += 1
        else:
            break
    content_len = len(detok_text) - trailing_dots  # length w/o trailing dots

    # --- dotted rounded border helpers ---
    def draw_dotted_rounded_border(draw, bbox, radius, color, spacing, r):
        x1, y1, x2, y2 = bbox
        w, h = max(0, x2 - x1), max(0, y2 - y1)
        if w == 0 or h == 0:
            return
        rad = max(0, min(radius, w // 2, h // 2))

        def dot(x, y):
            draw.ellipse((x - r, y - r, x + r, y + r), fill=color)

        xt1, xt2, yl1, yl2 = x1 + rad, x2 - rad, y1 + rad, y2 - rad
        x = xt1
        while x <= xt2:
            dot(x, y1)
            x += spacing
        y = yl1
        while y <= yl2:
            dot(x2, y)
            y += spacing
        x = xt2
        while x >= xt1:
            dot(x, y2)
            x -= spacing
        y = yl2
        while y >= yl1:
            dot(x1, y)
            y -= spacing
        step = max(4, int(spacing // 2))
        for a0, a1, cx, cy in [
            (180, 271, x1 + rad, y1 + rad),
            (270, 361, x2 - rad, y1 + rad),
            (0, 91, x2 - rad, y2 - rad),
            (90, 181, x1 + rad, y2 - rad),
        ]:
            for ang in range(a0, a1, step):
                a = math.radians(ang)
                dot(cx + rad * math.cos(a), cy + rad * math.sin(a))

    def draw_box(draw, bbox, fill, outline=None, width=1, radius=8, dotted=False):
        draw.rounded_rectangle(bbox, radius=radius, fill=fill)
        if outline is not None:
            if dotted:
                draw_dotted_rounded_border(
                    draw, bbox, radius, outline, dot_spacing, dot_radius
                )
            else:
                draw.rounded_rectangle(
                    bbox, radius=radius, outline=outline, width=width
                )

    # ---- Prompt text ----
    for line_idx, segment in enumerate(prompt_wrapped.split("\n")):
        toks = tokenizer.batch_decode(
            tokenizer(segment, return_tensors="pt")["input_ids"][0]
        )
        x = x_margin
        y = prompt_font.size * line_idx * line_height + (int(10 * s))
        for t in toks:
            w = int(d.textlength(t, font=prompt_font))
            d.text((x, y), t, (99, 99, 99), antialias=True, font=prompt_font)
            x += w

    # ---- Generated text ----
    word_counter = 0
    content_seen_global = 0  # counts non-dot tokens across lines

    for line_idx, segment in enumerate(wrapped_text.split("\n")):
        tokens = tokenizer.batch_decode(
            tokenizer(segment, return_tensors="pt")["input_ids"][0]
        )

        x = x_margin
        y = font.size * line_idx * line_height + (int(10 * s))
        _, yb_top, _, yb_bottom = d.textbbox((0, 0), "Ag", font=font)
        line_h = yb_bottom - yb_top
        y_top = int(y - y_padding)
        y_bottom = int(y + line_h + y_padding + extra_bottom)

        token_runs = []  # draw text after boxes
        earlier_x_min = earlier_x_max = None
        latest_x_min = latest_x_max = None

        for ind, word in enumerate(tokens):
            idx_global = word_counter + ind
            if idx_global >= len(detok_text):
                break

            use_font = bold_font if word == " ." else font

            # color logic
            is_answer = " ####" in tokens[:ind] and ind != len(tokens) - 1
            if detok_text[idx_global] == mask_token:
                color = box_fill  # hide masked token
            else:
                color = (28, 59, 87)
                if is_answer:
                    color = (0, 128, 0) if is_correct else (255, 0, 0)

            full_w = int(d.textlength(word, font=use_font))
            x_start, x_end = x, x + full_w

            # leading whitespace ignored for box span
            leading_ws = 0
            while leading_ws < len(word) and word[leading_ws] in (" ", "\t"):
                leading_ws += 1
            ws_w = (
                int(d.textlength(word[:leading_ws], font=use_font)) if leading_ws else 0
            )
            vis_x_start = x_start + ws_w
            vis_x_end = x_end

            is_dot = word == " ."

            # Only non-dots affect the latest/earlier spans and content index
            if not is_dot:
                content_seen_global += 1
                is_latest = content_seen_global > max(0, content_len - block_size)
                if vis_x_end > vis_x_start:
                    if is_latest:
                        latest_x_min = (
                            vis_x_start
                            if latest_x_min is None
                            else min(latest_x_min, vis_x_start)
                        )
                        latest_x_max = (
                            vis_x_end
                            if latest_x_max is None
                            else max(latest_x_max, vis_x_end)
                        )
                    else:
                        earlier_x_min = (
                            vis_x_start
                            if earlier_x_min is None
                            else min(earlier_x_min, vis_x_start)
                        )
                        earlier_x_max = (
                            vis_x_end
                            if earlier_x_max is None
                            else max(earlier_x_max, vis_x_end)
                        )

            token_runs.append((x_start, word, use_font, color))
            x = x_end

        # ---- Draw boxes FIRST (background) ----
        if earlier_x_min is not None and earlier_x_max is not None:
            ebbox = (
                int(earlier_x_min - x_padding),
                y_top,
                int(earlier_x_max + x_padding),
                y_bottom,
            )
            draw_box(d, ebbox, fill=box_fill, outline=None, radius=corner_radius)

        if latest_x_min is not None and latest_x_max is not None:
            lbbox = (
                int(latest_x_min - x_padding),
                y_top,
                int(latest_x_max + x_padding),
                y_bottom,
            )
            if is_enc_dec:
                if caching_step:
                    draw_box(
                        d,
                        lbbox,
                        fill=box_fill,
                        outline=border_color,
                        width=border_width_thin,
                        radius=corner_radius,
                        dotted=False,
                    )
                else:
                    draw_box(
                        d,
                        lbbox,
                        fill=box_fill,
                        outline=border_color,
                        width=border_width_thin,
                        radius=corner_radius,
                        dotted=True,
                    )
            else:
                draw_box(
                    d,
                    lbbox,
                    fill=box_fill,
                    outline=border_color,
                    width=border_width_thick,
                    radius=corner_radius,
                    dotted=False,
                )

        # ---- Draw TEXT AFTER (on top) ----
        for x_start, word, use_font, color in token_runs:
            d.text((x_start, y), word, color, antialias=True, font=use_font)

        word_counter += len(tokens)

    # Downscale if supersampling requested
    if downscale_output:
        img = img.resize((canvas_w, canvas_h), Image.LANCZOS)

    return img


def to_gif_palette(im):
    """
    Convert RGB frame to an 8-bit adaptive palette for GIF with dithering.
    This improves perceived quality and reduces banding.
    """
    return im.convert(
        "P", palette=Image.ADAPTIVE, colors=256, dither=Image.FLOYDSTEINBERG
    )


def create_gif(
    text_list,
    output_file="output.gif",
    font_size=25,  # kept for API compat (draw_image uses scale)
    duration=1,  # unused; per-frame durations handled via wall_clocks
    is_correct=True,
    is_enc_dec=True,
    block_size=4,
    caching_wall_clock=1.0,
    decoding_wall_clock=0.5,
    mask_token="<|fim_middle|>",
    scale=2,  # render scale for higher resolution
    canvas_width=1100,  # logical width at scale=1
    canvas_height=150,  # logical height at scale=1
    downscale_output=False,  # True = supersample then return base (canvas_*) size
):
    prompt = " "
    block_size = int(block_size)

    # --- Simple preprocess: strip endoftext markers
    def preprocess(text_list):
        return [t.replace("<|endoftext|>", " ") for t in text_list]

    text_list = preprocess(text_list)

    # append three trailing frames with ' .' (boxing logic ignores these for spans)
    text_list.append(text_list[-1] + " .")
    text_list.append(text_list[-1] + " .")
    text_list.append(text_list[-1] + " .")

    images = []
    wall_clocks = []

    # Wrap width derived from pixel width (coarse but stable)
    clean_text = "".join(text_list[-1]).replace("\n", " ")
    avg_char_px = max(6, int(10 * scale))
    wrap_cols = max(
        30, int(canvas_width * (scale if not downscale_output else 1)) // avg_char_px
    )
    wrapped_text = textwrap.fill(clean_text, width=wrap_cols)

    # Dimensions sent to draw_image (base or supersampled)
    target_w = int(canvas_width * (scale if not downscale_output else 1))
    target_h = int(canvas_height * (scale if not downscale_output else 1))

    parallel_chance = 0.25

    for sample_id, text_ in tqdm(enumerate(text_list)):
        # optional skip resembling "parallel" updates
        if random.random() < parallel_chance and mask_token in text_:
            continue

        detok_text = tokenizer.batch_decode(
            tokenizer(text_, return_tensors="pt")["input_ids"][0]
        )
        block_text = detok_text[-block_size:]

        # decoding frame
        if sample_id < len(text_list) - 3:
            img = draw_image(
                prompt,
                text_,
                wrapped_text,
                is_correct,
                is_enc_dec,
                block_size=block_size,
                img_width=target_w,
                img_height=target_h,
                mask_token=mask_token,
                scale=scale,
                downscale_output=downscale_output,
            )
            images.append(img)
            wall_clocks.append(decoding_wall_clock)

        # caching frame
        if mask_token not in block_text and is_enc_dec:
            img = draw_image(
                prompt,
                text_,
                wrapped_text,
                is_correct,
                is_enc_dec,
                block_size=block_size,
                img_width=target_w,
                img_height=target_h,
                caching_step=True,
                mask_token=mask_token,
                scale=scale,
                downscale_output=downscale_output,
            )
            images.append(img)
            wall_clocks.append(caching_wall_clock)

    # Convert frames to GIF palette for higher quality
    images_for_gif = [to_gif_palette(im) for im in images]

    # Build duration list (ms) aligned with frames
    durations_ms = []
    n = len(images_for_gif)
    for i in range(n):
        if i < n - 4:
            durations_ms.append(int(wall_clocks[i] * 40_000))
        else:
            durations_ms.append(1000)

    # Save GIF
    images_for_gif[0].save(
        output_file,
        save_all=True,
        append_images=images_for_gif[1:],
        optimize=True,
        duration=durations_ms,
        loop=0,
        disposal=2,
    )

In [ ]:
tokenizer.add_tokens("<|fim_middle|>")
create_gif(
    intm_samples_e2d2["intermediate_samples"][:67],
    output_file="e2d2.gif",
    is_correct=True,
    is_enc_dec=True,
    caching_wall_clock=0.009590480684295412,
    decoding_wall_clock=(0.017220060949250465 / 4),
)

0it [00:00, ?it/s]

64it [00:01, 43.55it/s] 

In [ ]:
tokenizer.add_tokens(["[PAD]"])
create_gif(
    intm_samples_bd3lm["intermediate_samples"][:71],
    output_file="bd3lm.gif",
    is_correct=False,
    is_enc_dec=False,
    caching_wall_clock=0.011140805522287923,
    decoding_wall_clock=(0.051167509844922646 / 4),
    mask_token="[PAD]",
)

74it [00:01, 67.68it/s] 
